In [3]:
# import tensorflow as tf
# from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np
import pickle

In [2]:
import torch
import torch.nn as nn

class AnnModel(nn.Module):
    def __init__(self, input_dim=12): 
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 8),
            nn.ReLU(),
            nn.Linear(8, 4),
            nn.ReLU(),
            nn.Linear(4, 1),
            nn.Sigmoid()
        )
    def forward(self, features):
        return self.network(features)

In [4]:
model = AnnModel()
model.load_state_dict(torch.load('ann_model_weights.pth'))
model.eval()

AnnModel(
  (network): Sequential(
    (0): Linear(in_features=12, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=4, bias=True)
    (3): ReLU()
    (4): Linear(in_features=4, out_features=1, bias=True)
    (5): Sigmoid()
  )
)

In [22]:
with open('label_encoder_geography.pkl','rb') as file:
    label_encoder_geo = pickle.load(file)

In [23]:
with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)


In [24]:
with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [47]:
input_data = {
    'CreditScore': 619,
    'Geography': 'France',
    'Gender': 'Female',
    'Age': 42,
    'Tenure': 2,
    'Balance': 0,
    'NumOfProducts': 1,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 101348.88
}

In [48]:
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]])
geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

/home/aviral-linux/GenAi/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [49]:
input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,France,Female,42,2,0,1,1,1,101348.88


In [50]:
input_df['Gender']=label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,619,France,0,42,2,0,1,1,1,101348.88


In [51]:
input_df=pd.concat([input_df.drop("Geography",axis=1),geo_encoded_df],axis=1)
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0,1,1,1,101348.88,1.0,0.0,0.0


In [52]:
input_scaled=scaler.transform(input_df)
input_scaled

array([[-0.33880827, -1.09499335,  0.29493847, -1.04241787, -1.21847056,
        -0.91668767,  0.64920267,  0.97481699,  0.01595384,  1.00150113,
        -0.57946723, -0.57638802]])

In [53]:
input_tensor = torch.tensor(input_scaled)
input_tensor_data = input_tensor.view(1,-1).float()

In [54]:
model.eval()
with torch.no_grad():
    predction = model(input_tensor_data)

In [55]:
result = predction.item()
print(result)

0.4059143364429474


In [57]:
if(result>0.35):
    print(f"predction: Likely to churn ({result:.2%} probability)")
else:
    print(f"Prediction: Likely to stay ({1 - result:.2%} probability)")
    



predction: Likely to churn (40.59% probability)
